# Tile-card pipeline (Colab)
اسم الشخصية ← Gemini يصمّم ← الكود يحوّل لبلاطات من مكتبتك ← رسم الكارت ← Gemini يراجع الصورة ← أحسن نسخة.

**قبل ما تبدئي:** (1) خدي API key مجاني من https://aistudio.google.com/apikey  (2) جهّزي ملف `tilesys_pkg.zip`.
شغّلي الخلايا بالترتيب.

In [ ]:
# 1) تثبيت المكتبات
!pip -q install cairosvg google-genai

In [ ]:
# 2) ارفعي tilesys_pkg.zip هنا
from google.colab import files
import os, zipfile
up = files.upload()                       # اختاري tilesys_pkg.zip
name = list(up.keys())[0]
os.makedirs('/content/tilesys', exist_ok=True)
zipfile.ZipFile(name).extractall('/content/tilesys')
os.chdir('/content/tilesys')
print(os.listdir('.'))

In [ ]:
# 3) مفتاح الـ API (بيتكتب مخفي)
from getpass import getpass
API_KEY = getpass('Gemini API key: ')

In [ ]:
# 4) الأساسيات: المكتبة + الـ solver + الـ renderer
import json, re, io, time, sys, os
sys.path.insert(0, '.')
import cairosvg
import solver                                         # solver.py من الحزمة
CAT  = json.load(open('catalog.json'))
TILE = {int(k): v['d'] for k, v in CAT['tiles'].items()}
ORI  = {o['key']: o for o in CAT['orients']}
SYS1 = open('SYSTEM_PROMPT_1_silhouette.md', encoding='utf-8').read()     # الموديل يرسم silhouette
SYS2 = open('SYSTEM_PROMPT_2_direct_tiles.md', encoding='utf-8').read()   # الموديل يختار البلاطات
GOLD = json.load(open('gold.json'))                   # أمثلتك البشرية (خمس تصاميم)
SYS2 += "\n\nHUMAN-DESIGNED GOLD EXAMPLES (style reference ONLY: note how few features are exaggerated, how triangles/quarter-discs shape the outline, how tile types are VARIED. Never copy a layout):\n" + \
        "\n".join(json.dumps(g) for g in GOLD.values())

def cell_tf(o):
    t = ''
    if o['rot']:  t += f"rotate({o['rot']} 50 50) "
    if o['flip']: t += 'translate(100 0) scale(-1 1)'
    return t.strip()

def valid_grid(g):
    if not (isinstance(g, list) and len(g) == 5 and all(isinstance(r, list) and len(r) == 5 for r in g)): return False
    return all(k in ('.', '#') or k in ORI for r in g for k in r)

def grid_group(g, fill='#111', stroke=None, sw=2):
    s = ''
    for r in range(5):
        for c in range(5):
            k = g[r][c]; inner = ''
            if k == '#':
                inner = f'<rect width="100" height="100" fill="{fill}"/>'
            elif k != '.':
                o = ORI[k]
                st = f' stroke="{stroke}" stroke-width="{sw}" stroke-linejoin="round"' if stroke else ''
                inner = f'<g transform="{cell_tf(o)}"><path d="{TILE[o["tile"]]}" fill="{fill}" fill-rule="evenodd"{st}/></g>'
            s += f'<g transform="translate({c*100} {r*100})">{inner}</g>'
    return s

def lines_svg(x0, y0, cs, color='#9aa1a8', w=1.5):
    out = ''
    for i in range(6):
        out += f'<line x1="{x0+i*cs}" y1="{y0}" x2="{x0+i*cs}" y2="{y0+5*cs}" stroke="{color}" stroke-width="{w}"/>'
        out += f'<line x1="{x0}" y1="{y0+i*cs}" x2="{x0+5*cs}" y2="{y0+i*cs}" stroke="{color}" stroke-width="{w}"/>'
    return out

def card_svg(g, name_ar='', name_en=''):
    W, H, GP, GS, M = 400, 540, 30, 340, 10
    cs = (GS - 2*M) / 5; k = cs / 100
    return (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="{W}" height="{H}" font-family="Cairo, Arial, sans-serif">'
      f"<style>@import url('https://fonts.googleapis.com/css2?family=Cairo:wght@800&amp;display=swap');</style>"
      f'<rect width="{W}" height="{H}" rx="30" fill="#58C3DF"/><rect x="{GP}" y="{GP}" width="{GS}" height="{GS}" rx="6" fill="#fff"/>'
      f'<g transform="translate({GP+M} {GP+M}) scale({k})">{grid_group(g)}</g>{lines_svg(GP+M, GP+M, cs)}'
      f'<g transform="translate({W/2-25} 386) scale(0.1)">{grid_group(g, fill="#fff", stroke="#14566f", sw=9)}</g>'
      f'<text x="{W/2}" y="486" text-anchor="middle" font-size="40" font-weight="800" fill="#0f2a3a" direction="rtl">{name_ar}</text>'
      f'<text x="{W/2}" y="520" text-anchor="middle" font-size="24" font-weight="800" fill="#0f2a3a" letter-spacing="1">{name_en.upper()}</text></svg>')

def grid_png(g):
    """صورة الشبكة بس (من غير نص) عشان موديل المراجعة"""
    svg = (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="-10 -10 520 520" width="520" height="520">'
           f'<rect x="-10" y="-10" width="520" height="520" fill="#fff"/>{grid_group(g)}{lines_svg(0, 0, 100, w=2)}</svg>')
    return cairosvg.svg2png(bytestring=svg.encode())

In [ ]:
# 5) الاتصال بـ Gemini
from google import genai
from google.genai import types
import concurrent.futures as cf
client = genai.Client(api_key=API_KEY)

INCLUDE_LITE = False
# --- اكتشاف موديلات Gemini 3 المتاحة لمفتاحك (flash الأول ثم lite ثم pro)
def discover_models():
    names = []
    for m in client.models.list():
        n = m.name.replace('models/', '')
        acts = getattr(m, 'supported_actions', None) or []
        if n.startswith('gemini') and ('generateContent' in acts or not acts) \
           and not any(x in n for x in ['tts', 'image', 'embedding', 'live', 'audio', 'robotics', 'computer', 'learnlm', 'aqa']):
            names.append(n)
    if not INCLUDE_LITE: names = [n for n in names if 'lite' not in n]   # lite بيطلّع تصاميم ضعيفة
    g3 = [n for n in names if n.startswith('gemini-3')]
    names = g3 or names                                   # Gemini 3 بس لو متاح
    def key(n):
        ver = re.findall(r'(\\d+(?:\\.\\d+)?)', n)
        v = float(ver[0]) if ver else 0
        tier = 0 if 'flash' in n and 'lite' not in n else (1 if 'lite' in n else 2)
        return (tier, -v, n)
    return sorted(set(names), key=key)

INCLUDE_LITE = False    # True = يسمح بموديلات lite (أضعف جداً في التصميم)
FALLBACK_MODELS = ['gemini-3-flash-preview', 'gemini-3-pro-preview']   # لو الاكتشاف التلقائي فشل
MY_MODELS = []          # لو عايزة تفرضي ترتيب معين اكتبيه هنا
LOW_THINK = False       # False = جودة أعلى (تفكير كامل). True = أسرع بس تصاميم أضعف
WORKERS = 2             # كام تصميم يتعملوا في نفس الوقت (زوّديه لو الحد المجاني بيسمح)
try:
    MODEL_LIST = MY_MODELS or discover_models()
except Exception as e:
    print('تعذّر اكتشاف الموديلات:', str(e)[:150]); MODEL_LIST = MY_MODELS
if not MODEL_LIST: MODEL_LIST = FALLBACK_MODELS
print('الموديلات اللي هجرّبها بالترتيب:', MODEL_LIST[:10])
BAD = set()

def make_cfg(system, use_json, low):
    kw = dict(system_instruction=system, temperature=1.0)
    if use_json: kw['response_mime_type'] = 'application/json'
    if low:
        try: kw['thinking_config'] = types.ThinkingConfig(thinking_level='low')
        except Exception: pass
    return types.GenerateContentConfig(**kw)

def ask(system, parts, json_out=True, rounds=3):
    t0 = time.time()
    for rnd in range(rounds):
        for m in [x for x in MODEL_LIST if x not in BAD]:
            use_json, low = json_out, LOW_THINK
            for _ in range(3):
                try:
                    r = client.models.generate_content(model=m, contents=parts, config=make_cfg(system, use_json, low))
                    print(f'    [{m}] {time.time() - t0:.0f}s')
                    return r.text
                except Exception as e:
                    msg = str(e); lw = msg.lower()
                    if low and 'thinking' in lw: low = False; continue          # الموديل مابيدعمش تقليل التفكير
                    if use_json and ('mime' in lw or 'json mode' in lw): use_json = False; continue
                    if '404' in msg or 'not_found' in lw or 'not found' in lw or 'not supported' in lw or '403' in msg or 'permission' in lw:
                        BAD.add(m); print('  تخطّي', m, '(مش متاح)'); break
                    if '429' in msg or 'resource_exhausted' in lw or 'quota' in lw:
                        if 'limit: 0' in lw: BAD.add(m); print('  تخطّي', m, '(مش متاح في الباقة المجانية)')
                        else:
                            mm = re.search(r'retry in ([\d.]+)s|retryDelay[^\d]*(\d+)', msg)
                            wait = min(float(next(g for g in mm.groups() if g)) + 1, 65) if mm else 20
                            print('  ', m, f'وصل للحد المؤقت ← بستنى {wait:.0f}ث وبرجع لنفس الموديل (مش هنزل لموديل أضعف)'); time.sleep(wait); continue
                        break
                    print('  ', m, 'خطأ مؤقت:', msg[:100]); time.sleep(4)
        print('  كل الموديلات مشغولة، هستنى 20 ثانية...'); time.sleep(20)
    raise RuntimeError('كل موديلات Gemini فشلت. جرّبي بعد شوية أو افحصي المفتاح')

In [ ]:
# 6) الـ pipeline: تصميم ← بلاطات ← مراجعة ← أحسن نسخة
def parse_json(t):
    t = t.strip()
    t = re.sub(r'^```(?:json)?|```$', '', t, flags=re.M).strip()
    return json.loads(t[t.index('{'): t.rindex('}') + 1])

BLIND = """You see a 5x5 grid of black geometric tiles forming a simple flat icon. WITHOUT any hint, say what it depicts.
Use the simplest everyday name (cat, bird, fish, dog, rabbit, owl, car, boat... not sparrow / kitten).
Reply with JSON only: {"guesses": ["most likely", "second guess", "third guess"], "description": "one short sentence: what you actually see (shapes, which parts are missing or unclear)"}"""

ADVISE = """A 5x5 tile icon was meant to depict a {name} but a viewer saw: {desc}
Look at the image and give ONE concrete fix to make it clearly a {name} (which cells/features to add, remove or move). Reply JSON only: {{"fix": "..."}}"""

import random
HISTORY = {}
HINTS = ['side view facing left', 'side view facing right', 'front view, symmetric', 'three-quarter view, exaggerate the head',
         'emphasise the tail/back', 'use many diagonal triangles for the outline', 'use curved quarter-discs for a rounder body',
         'use a different tile family than usual: try the less common tiles from the catalog']
def quality(grid):
    """تقييم تلقائي: تنوّع البلاطات، عدد الخانات المليانة، وعدم وجود قطع عايمة."""
    cells = [(r, c, grid[r][c]) for r in range(5) for c in range(5) if grid[r][c] not in ('.', '')]
    n = len(cells); kinds = len({k for _, _, k in cells}); s = 0.0
    s += min(kinds, 7) / 7 * 0.4
    s += 0.3 if 9 <= n <= 18 else 0.0
    pos = {(r, c) for r, c, _ in cells}
    floating = sum(1 for r, c in pos if not any((r + dr, c + dc) in pos for dr, dc in ((1,0),(-1,0),(0,1),(0,-1))))
    s += 0.3 if floating == 0 else 0.0
    full = sum(1 for _, _, k in cells if k == '0a')
    if full > 9: s -= 0.3
    return s

def png_part(grid): return types.Part.from_bytes(data=grid_png(grid), mime_type='image/png')

def recognise(name_en, grid):
    """اختبار تعرّف أعمى: الموديل مابيعرفش الاسم. لو خمّن الاسم الصح يبقى الشكل واضح فعلاً."""
    out = parse_json(ask(BLIND, [png_part(grid), 'What does this icon depict?']))
    gs = [str(g).lower().strip() for g in out.get('guesses', [])]
    key = name_en.lower().strip()
    rank = next((i for i, g in enumerate(gs) if key in g or (g and g in key)), None)
    score = {0: 10, 1: 7, 2: 5}.get(rank, 1)
    return float(score), out.get('description', '') + '  | guesses: ' + ', '.join(gs)

def advise(name_en, grid, desc):
    out = parse_json(ask(ADVISE.format(name=name_en, desc=desc), [png_part(grid), 'Suggest the fix.']))
    return out.get('fix', '')

def design(name, variation, mode, feedback=None, prev_grid=None):
    msg = {"character": name, "variation": random.choice(HINTS) + f' (seed {random.randint(1,9999)})'}
    old = HISTORY.get(name.lower(), [])[-4:]
    if old: msg["avoid_these_previous_layouts"] = old
    if feedback: msg["feedback_on_previous_attempt"] = feedback
    parts = [json.dumps(msg, ensure_ascii=False)]
    if prev_grid: parts = [png_part(prev_grid), 'This is your previous attempt (rendered). ' + parts[0]]
    for _try in range(2):                                   # لو الـ JSON باظ نحاول مرة تانية
        try: d = parse_json(ask(SYS1 if mode == 'silhouette' else SYS2, parts)); break
        except (ValueError, KeyError) as e:
            if _try: raise
    if mode == 'silhouette':
        grid, _ = solver.solve(solver.rast(d['svg']))
    else:
        grid = d['grid']
    if not valid_grid(grid): raise ValueError('invalid grid from model')
    return {'mode': mode, 'name_en': d.get('name_en', name), 'name_ar': d.get('name_ar', ''), 'grid': grid}

from PIL import Image, ImageDraw
JUDGE = """You are judging {n} candidate 5x5 geometric tile icons of a "{name}", shown side by side and numbered 1..{n}.
Rank them from the BEST to the WORST icon of a {name}. Judge like a child shown the icon: which one is clearest and most clearly a {name} and not some other animal/object?
Penalise: looks like another thing (arrow, rocket, cat instead of owl...), missing signature feature, noisy/random tiles, floating bits, a solid blob.
Reply JSON only: {{"ranking": [best_number, ..., worst_number], "why": "one short sentence on the winner"}}"""

def sheet(cands):
    ims = [Image.open(io.BytesIO(grid_png(c['grid']))).convert('RGB').resize((360, 360)) for c in cands]
    W = Image.new('RGB', (370 * len(ims), 400), 'white'); d = ImageDraw.Draw(W)
    for i, im in enumerate(ims):
        W.paste(im, (i * 370 + 5, 38)); d.text((i * 370 + 160, 8), f'#{i + 1}', fill='black', font_size=26) if hasattr(d, 'text') else None
    b = io.BytesIO(); W.save(b, 'PNG'); return types.Part.from_bytes(data=b.getvalue(), mime_type='image/png')

def judge(name_en, cands):
    """حكم مقارن: الموديل يشوف كل التصاميم جنب بعض ويرتّبها (أدق بكتير من درجة مطلقة لكل واحد)"""
    out = parse_json(ask(JUDGE.format(n=len(cands), name=name_en), [sheet(cands), 'Rank them.']))
    order = [int(x) - 1 for x in out.get('ranking', []) if 0 < int(x) <= len(cands)]
    for pos, i in enumerate(order): cands[i]['score'] += {0: 4, 1: 2.5, 2: 1}.get(pos, 0)
    return out.get('why', '')

GOLD_USER = json.load(open('gold_user.json')) if os.path.exists('gold_user.json') else []
for _g in GOLD_USER: SYS2 += '\nUSER-APPROVED EXAMPLE (best style, never copy the layout): ' + json.dumps(_g)
def like(cands, i):
    """اختاري الأحسن بنفسك: بيتحفظ كمثال والموديل بيتعلم منه في المرات الجاية"""
    global SYS2
    g = cands[i - 1]['grid']; GOLD_USER.append(g)
    SYS2 += '\nUSER-APPROVED EXAMPLE (best style, never copy the layout): ' + json.dumps(g)
    json.dump(GOLD_USER, open('gold_user.json', 'w')); print('اتحفظ كمثال ✔ (', len(GOLD_USER), ')')

def make_card(name, n=4, refine=True, target=7):
    t0 = time.time()
    def one(k):
        mode = 'silhouette' if k % 2 == 0 else 'direct'      # نجرّب الطريقتين
        c = design(name, k + 1, mode)
        c['score'], c['problems'] = recognise(c['name_en'], c['grid'])
        c['score'] += quality(c['grid']) * 0.3
        return c
    cands = []
    with cf.ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = [ex.submit(one, k) for k in range(n)]
        for k, f in enumerate(futs):
            try:
                c = f.result(); cands.append(c)
                print(f'  {c["mode"]:10s} #{k+1}: score {c["score"]}  {c["problems"][:90]}')
            except Exception as e:
                print(f'  #{k+1} failed:', str(e)[:100])
    if not cands: raise RuntimeError('no valid candidate')
    if len(cands) >= 2:
        try: print('  الحكم:', judge(cands[0]['name_en'], cands))
        except Exception as e: print('  judge failed:', str(e)[:100])
    best = max(cands, key=lambda c: c['score'])
    if refine and best['score'] < target:                      # جولة تصحيح واحدة
        try:
            fix = advise(best['name_en'], best['grid'], best['problems'])
            c = design(name, 99, best['mode'], feedback='It looked like: ' + best['problems'] + ' FIX: ' + fix, prev_grid=best['grid'])
            c['score'], c['problems'] = recognise(c['name_en'], c['grid']); c['score'] += quality(c['grid']) * 0.3
            cands.append(c); print(f'  refined: score {c["score"]}')
            if c['score'] > best['score']: best = c
        except Exception as e:
            print('  refine failed:', str(e)[:100])
    HISTORY.setdefault(name.lower(), []).append(best['grid'])
    print(f'  الوقت الكلي: {time.time() - t0:.0f} ثانية')
    return best, cands

from IPython.display import HTML, display
def show(best, cands):
    html = '<div style="display:flex;flex-wrap:wrap;gap:12px;align-items:flex-start">'
    html += f'<div><b>الأفضل (score {best["score"]})</b><br>{card_svg(best["grid"], best["name_ar"], best["name_en"])}</div>'
    for i, c in enumerate(cands, 1):
        html += f'<div style="transform:scale(.6);transform-origin:top left;width:240px;height:330px"><small>#{i} · {c["mode"]} · {c["score"]:.1f}</small><br>{card_svg(c["grid"], c["name_ar"], c["name_en"])}</div>'
    display(HTML(html + '</div>'))

In [ ]:
# 7) جرّبي! غيّري الأسماء
for name in ['عصفور', 'Cat', 'Fish']:
    print('==', name)
    best, cands = make_card(name, n=4)
    show(best, cands)
    print(json.dumps(best['grid']))      # الشبكة نفسها لو عايزة تحفظيها
# لو واحد من التصاميم المعروضة (#1, #2...) عاجبك أكتر من "الأفضل": like(cands, 2)  ← بيتعلّم منه

In [ ]:
# 8) تثبيت أداة حفظ PNG (مرة واحدة، تاخد دقيقة أو اتنين)
!pip -q install playwright
!playwright install --with-deps chromium

In [ ]:
# 9) دوال الحفظ والتحميل
import os, subprocess, shutil, re
from google.colab import files
os.makedirs('cards', exist_ok=True)

SHOT = "import sys\n" \
"from playwright.sync_api import sync_playwright\n" \
"svg = open(sys.argv[1], encoding='utf-8').read()\n" \
"with sync_playwright() as p:\n" \
"    b = p.chromium.launch(args=['--no-sandbox'])\n" \
"    pg = b.new_page(viewport={'width': 400, 'height': 540}, device_scale_factor=3)\n" \
"    pg.set_content('<html><body style=\"margin:0;background:transparent\">' + svg + '</body></html>')\n" \
"    pg.wait_for_timeout(2500)\n" \
"    pg.screenshot(path=sys.argv[2], omit_background=True)\n" \
"    b.close()\n"
open('shot.py', 'w').write(SHOT)

def save_card(best, png=True):
    # يحفظ الكارت: SVG + PNG + ملف الشبكة (JSON) جوه فولدر cards
    base = re.sub(r'[^A-Za-z0-9]+', '_', best['name_en']).strip('_') or 'card'
    base = f"cards/{base}_{int(time.time()) % 100000}"
    open(base + '.svg', 'w', encoding='utf-8').write(card_svg(best['grid'], best['name_ar'], best['name_en']))
    json.dump({'name_en': best['name_en'], 'name_ar': best['name_ar'], 'grid': best['grid'], 'score': best['score']},
              open(base + '.json', 'w', encoding='utf-8'), ensure_ascii=False)
    if png:
        r = subprocess.run(['python', 'shot.py', base + '.svg', base + '.png'], capture_output=True, text=True)
        if r.returncode: print('  PNG فشل:', r.stderr[-200:])
    print('  اتحفظ:', base)
    return base

def download_all():
    shutil.make_archive('cards_all', 'zip', 'cards'); files.download('cards_all.zip')   # كل الكروت في zip واحد

In [ ]:
# 10) احفظي كل كارت بعد ما يتعمل، وبعدين نزّلي الكل
for name in ['Owl', 'Rabbit']:
    best, cands = make_card(name, n=4)
    show(best, cands)
    save_card(best)
download_all()          # بينزّل zip فيه كل الصور (png و svg و json)
# لتحميل كارت واحد:  files.download('cards/اسم_الملف.png')

## ملاحظات
- لو ظهر `429` أو `quota` : الحد المجاني خلص، استني شوية أو قلّلي `n`.
- كل شخصية = حوالي 8–10 نداءات للموديل (4 تصميمات + 4 اختبارات تعرّف أعمى + جولة تصحيح لو لزم). لو عايزة أسرع: `make_card(name, n=2, refine=False)`.
- الموديلات بتتكشف تلقائياً حسب مفتاحك، ولو موديل مش متاح أو وصل للحد بيجرّب اللي بعده. لفرض ترتيب معين اكتبيه في `MY_MODELS` في خلية 5، ولتسريع الشغل `LOW_THINK = True` و`WORKERS`.
- ولو عايزة Claude أو GPT غيّري خلية 5 بس (دالة `ask`).